# Part 5: Baseline Benchmarking (Kaggle Environment)

This notebook covers:
- Setting up GPU acceleration environment (T4/P100)
- Benchmarking Qwen2.5-7B-Instruct and Mistral-7B-Instruct-v0.3 (zero-shot, 4-bit quantized) on Financial PhraseBank
- Comparative performance evaluation against fine-tuned FinBERT and the Part 4 multi-agent pipeline
- Exporting benchmark metrics to `results/metrics/`

**Prerequisite (Kaggle Settings -> Secrets):** add your `HF_TOKEN` as a Kaggle secret — Mistral-7B-Instruct-v0.3 is gated and requires it.

## 1. Environment setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers==4.46.3 bitsandbytes==0.50.2 accelerate scikit-learn tqdm

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## 2. Clone repo and pull in Financial PhraseBank

Cloning fresh so this notebook runs standalone on Kaggle without depending on any other notebook's local state.

In [ ]:
!git clone -q https://github.com/samarthghag/retail-sentiment-micro-agents.git /kaggle/working/repo
%cd /kaggle/working/repo
import sys
sys.path.insert(0, "/kaggle/working/repo")

In [ ]:
from src.data_loading import load_financial_phrasebank
from config.global_config import CONFIG

fpb_splits = load_financial_phrasebank(
    data_path=CONFIG["data_dir"],
    agreement_threshold=CONFIG["fpb_config"],
)
test_df = fpb_splits["test"]
print(f"FPB test split: {len(test_df)} sentences")
test_df["label_name"].value_counts()

## 3. Run Qwen2.5-7B-Instruct baseline (one-shot, 4-bit)

Rerun with an updated prompt: the first zero-shot pass had a 19.4% `unparseable_rate` (Qwen2.5-Instruct tended to wander off-format), so this version adds one worked example and tightens `max_new_tokens` to 4. Note this makes the run **one-shot**, not zero-shot — Mistral's result (below, already 0.0% unparseable) stays zero-shot, so report the two with that distinction made explicit rather than as a like-for-like "both zero-shot" comparison.

In [ ]:
from src.baselines.qwen_baseline import QwenBaselineEvaluator

qwen_evaluator = QwenBaselineEvaluator(model_name=CONFIG["monolithic_baseline_a"], load_in_4bit=CONFIG["load_in_4bit"])
qwen_metrics = qwen_evaluator.run_benchmark(test_df, batch_size=8)
qwen_metrics

In [ ]:
qwen_evaluator.unload()  # free GPU memory before loading Mistral

## 4. Run Mistral-7B-Instruct-v0.3 baseline (zero-shot, 4-bit)

In [ ]:
from src.baselines.mistral_baseline import MistralBaselineEvaluator

mistral_evaluator = MistralBaselineEvaluator(model_name=CONFIG["monolithic_baseline_b"], load_in_4bit=CONFIG["load_in_4bit"])
mistral_metrics = mistral_evaluator.run_benchmark(test_df, batch_size=8)
mistral_metrics

In [ ]:
mistral_evaluator.unload()

## 5. Save metrics and compare against FinBERT

No numbers are hardcoded here — everything is read back from the JSON files that were
just written, so what prints below is guaranteed to match what's on disk.

In [ ]:
import json
from pathlib import Path

metrics_dir = Path(CONFIG["processed_dir"]).parent.parent / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

with open(metrics_dir / "qwen_baseline_metrics.json", "w") as f:
    json.dump(qwen_metrics, f, indent=2)

with open(metrics_dir / "mistral_baseline_metrics.json", "w") as f:
    json.dump(mistral_metrics, f, indent=2)

print("Saved to:", metrics_dir)
!ls -la {metrics_dir}

In [ ]:
import pandas as pd

finbert_path = metrics_dir / "finbert_test_metrics.json"
rows = []
if finbert_path.exists():
    with open(finbert_path) as f:
        rows.append({"model": "FinBERT (fine-tuned)", **json.load(f)})

with open(metrics_dir / "qwen_baseline_metrics.json") as f:
    rows.append({"model": "Qwen2.5-7B-Instruct (one-shot)", **json.load(f)})

with open(metrics_dir / "mistral_baseline_metrics.json") as f:
    rows.append({"model": "Mistral-7B-Instruct-v0.3 (zero-shot)", **json.load(f)})

comparison_df = pd.DataFrame(rows).set_index("model")
comparison_df